In [1]:
import glob
import random
import os

from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
import itertools


In [2]:
dataroot = 'CycleGAN/faces2paintings/'
epoch = 0
n_epochs = 20
batchSize = 2
lr = 0.0002
decay_epoch = 100
size = 256
input_nc = 3
output_nc = 3
cuda = torch.cuda.is_available()
n_cpu = 8

# Model Architecture

In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, in_features):
        super(ResidualBlock, self).__init__()

        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_features, in_features, 3),
            nn.InstanceNorm2d(in_features),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_features, in_features, 3),
            nn.InstanceNorm2d(in_features)
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, input_nc, output_nc, n_residual_blocks=9):
        super(Generator, self).__init__()

        # Initial convolution block
        model = [   nn.ReflectionPad2d(3),
                    nn.Conv2d(input_nc, 64, 7),
                    nn.InstanceNorm2d(64),
                    nn.ReLU(inplace=True) ]

        # Downsampling
        in_features = 64
        out_features = in_features*2
        for _ in range(2):
            model += [  nn.Conv2d(in_features, out_features, 3, stride=2, padding=1),
                        nn.InstanceNorm2d(out_features),
                        nn.ReLU(inplace=True) ]
            in_features = out_features
            out_features = in_features*2

        # Residual blocks
        for _ in range(n_residual_blocks):
            model += [ResidualBlock(in_features)]

        # Upsampling
        out_features = in_features//2
        for _ in range(2):
            model += [  nn.ConvTranspose2d(in_features, out_features, 3, stride=2, padding=1, output_padding=1),
                        nn.InstanceNorm2d(out_features),
                        nn.ReLU(inplace=True) ]
            in_features = out_features
            out_features = in_features//2

        # Output layer
        model += [  nn.ReflectionPad2d(3),
                    nn.Conv2d(64, output_nc, 7),
                    nn.Tanh() ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self, input_nc):
        super(Discriminator, self).__init__()

        # A bunch of convolutions one after another
        model = [   nn.Conv2d(input_nc, 64, 4, stride=2, padding=1),
                    nn.LeakyReLU(0.2, inplace=True) ]

        model += [  nn.Conv2d(64, 128, 4, stride=2, padding=1),
                    nn.InstanceNorm2d(128),
                    nn.LeakyReLU(0.2, inplace=True) ]

        model += [  nn.Conv2d(128, 256, 4, stride=2, padding=1),
                    nn.InstanceNorm2d(256),
                    nn.LeakyReLU(0.2, inplace=True) ]

        model += [  nn.Conv2d(256, 512, 4, padding=1),
                    nn.InstanceNorm2d(512),
                    nn.LeakyReLU(0.2, inplace=True) ]

        # FCN classification layer
        model += [nn.Conv2d(512, 1, 4, padding=1)]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        x =  self.model(x)
        # Average pooling and flatten
        return F.avg_pool2d(x, x.size()[2:]).view(x.size()[0], -1)

# --- 2. Dataset Loader ---

In [4]:
class ImageDataset(Dataset):
    def __init__(self, root, transforms_=None, unaligned=False, mode='train', max_images=None):
        self.transform = transforms.Compose(transforms_)
        self.unaligned = unaligned
        self.max_images = max_images

        all_files_A = sorted(glob.glob(os.path.join(root, f'{mode}A') + '/*.*'))
        all_files_B = sorted(glob.glob(os.path.join(root, f'{mode}B') + '/*.*'))

        if self.max_images:
            self.files_A = all_files_A[:self.max_images]
            self.files_B = all_files_B[:self.max_images]
        else:
            self.files_A = all_files_A
            self.files_B = all_files_B

        print(f"Dataset '{mode}A' loaded with {len(self.files_A)} images.")
        print(f"Dataset '{mode}B' loaded with {len(self.files_B)} images.")

    def __getitem__(self, index):
        image_A = Image.open(self.files_A[index % len(self.files_A)]).convert('RGB')
        
        if self.unaligned:
            image_B = Image.open(self.files_B[random.randint(0, len(self.files_B) - 1)]).convert('RGB')
        else:
            image_B = Image.open(self.files_B[index % len(self.files_B)]).convert('RGB')
        
        item_A = self.transform(image_A)
        item_B = self.transform(image_B)

        return {'A': item_A, 'B': item_B}

    def __len__(self):
        # The length of the dataset is the maximum of the two folders
        return max(len(self.files_A), len(self.files_B))



transforms_ = [
    transforms.Resize(int(size * 1.12), Image.BICUBIC),
    transforms.RandomCrop(size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
]

num_images_to_use = 10000

dataset = ImageDataset(dataroot,
                       transforms_=transforms_,
                       unaligned=True,
                       max_images=num_images_to_use)

dataloader = DataLoader(dataset,
                        batch_size=batchSize,
                        shuffle=True,
                        num_workers=n_cpu)

print(f"\nTraining with a subset of {num_images_to_use} images.")
print(f"Total batches in DataLoader: {len(dataloader)}")
print(len(dataloader))

Dataset 'trainA' loaded with 10000 images.
Dataset 'trainB' loaded with 8683 images.

Training with a subset of 10000 images.
Total batches in DataLoader: 5000
5000


# --- 3. Training Script ---

In [5]:
if not os.path.exists('outputs/A'):
    os.makedirs('outputs/A')
if not os.path.exists('outputs/B'):
    os.makedirs('outputs/B')
if not os.path.exists('saved_model'): 
    os.makedirs('saved_model')

In [6]:
netG_A2B = Generator(input_nc, output_nc)
netG_B2A = Generator(output_nc, input_nc)
netD_A = Discriminator(input_nc)
netD_B = Discriminator(output_nc)

if cuda:
    netG_A2B.cuda()
    netG_B2A.cuda()
    netD_A.cuda()
    netD_B.cuda()

In [7]:
criterion_GAN = torch.nn.MSELoss()
criterion_cycle = torch.nn.L1Loss()
criterion_identity = torch.nn.L1Loss()

optimizer_G = torch.optim.Adam(itertools.chain(netG_A2B.parameters(), netG_B2A.parameters()),
                                lr=lr, betas=(0.5, 0.999))
optimizer_D_A = torch.optim.Adam(netD_A.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D_B = torch.optim.Adam(netD_B.parameters(), lr=lr, betas=(0.5, 0.999))

In [ ]:
best_loss_G = float('inf')

print("Starting training loop...")
for epoch in range(epoch, n_epochs):
    # NEW: Variables to track the average loss for this epoch
    running_loss_G = 0.0
    running_loss_D = 0.0

    for i, batch in enumerate(dataloader):
        # Set model input
        real_A = batch['A'].cuda() if cuda else batch['A']
        real_B = batch['B'].cuda() if cuda else batch['B']

        # Adversarial ground truths
        target_real = torch.ones(batchSize, 1, requires_grad=False).cuda() if cuda else torch.ones(batchSize, 1, requires_grad=False)
        target_fake = torch.zeros(batchSize, 1, requires_grad=False).cuda() if cuda else torch.zeros(batchSize, 1, requires_grad=False)


        ###### Generators A2B and B2A ######
        optimizer_G.zero_grad()

        # Identity loss
        same_B = netG_A2B(real_B)
        loss_identity_B = criterion_identity(same_B, real_B) * 5.0
        same_A = netG_B2A(real_A)
        loss_identity_A = criterion_identity(same_A, real_A) * 5.0

        # GAN loss
        fake_B = netG_A2B(real_A)
        pred_fake = netD_B(fake_B)
        loss_GAN_A2B = criterion_GAN(pred_fake, target_real)

        fake_A = netG_B2A(real_B)
        pred_fake = netD_A(fake_A)
        loss_GAN_B2A = criterion_GAN(pred_fake, target_real)

        # Cycle loss
        recovered_A = netG_B2A(fake_B)
        loss_cycle_ABA = criterion_cycle(recovered_A, real_A) * 10.0

        recovered_B = netG_A2B(fake_A)
        loss_cycle_BAB = criterion_cycle(recovered_B, real_B) * 10.0

        # Total loss
        loss_G = loss_identity_A + loss_identity_B + loss_GAN_A2B + loss_GAN_B2A + loss_cycle_ABA + loss_cycle_BAB
        loss_G.backward()

        optimizer_G.step()

        ###### Discriminator A ######
        optimizer_D_A.zero_grad()
        pred_real = netD_A(real_A)
        loss_D_real = criterion_GAN(pred_real, target_real)
        pred_fake = netD_A(fake_A.detach())
        loss_D_fake = criterion_GAN(pred_fake, target_fake)
        loss_D_A = (loss_D_real + loss_D_fake) * 0.5
        loss_D_A.backward()
        optimizer_D_A.step()

        ###### Discriminator B ######
        optimizer_D_B.zero_grad()
        pred_real = netD_B(real_B)
        loss_D_real = criterion_GAN(pred_real, target_real)
        pred_fake = netD_B(fake_B.detach())
        loss_D_fake = criterion_GAN(pred_fake, target_fake)
        loss_D_B = (loss_D_real + loss_D_fake) * 0.5
        loss_D_B.backward()
        optimizer_D_B.step()
        
        # --- Update running losses ---
        running_loss_G += loss_G.item()
        running_loss_D += (loss_D_A + loss_D_B).item()

        if (i+1) % 100 == 0: # Print status every 100 batches
            print(f"Epoch [{epoch}/{n_epochs}], Batch [{i+1}/{len(dataloader)}], Loss D: {(loss_D_A + loss_D_B).item():.4f}, Loss G: {loss_G.item():.4f}")


    # Calculate average losses for the epoch
    avg_loss_G = running_loss_G / len(dataloader)
    avg_loss_D = running_loss_D / len(dataloader)
    
    print(f"\n--- End of Epoch {epoch} ---")
    print(f"Average Generator Loss: {avg_loss_G:.4f}")
    print(f"Average Discriminator Loss: {avg_loss_D:.4f}")
    print("--------------------------\n")

    # Save generated image examples
    fake_B_sample = 0.5 * (netG_A2B(real_A).data + 1.0)
    fake_A_sample = 0.5 * (netG_B2A(real_B).data + 1.0)
    save_image(fake_A_sample, f'outputs/A/epoch_{epoch}.png', nrow=5, normalize=True)
    save_image(fake_B_sample, f'outputs/B/epoch_{epoch}.png', nrow=5, normalize=True)
    
    if avg_loss_G < best_loss_G:
        best_loss_G = avg_loss_G
        print(f"** New best model found! Saving to 'saved_model' with G_loss: {best_loss_G:.4f} **\n")
        
        # Save generator models
        torch.save(netG_A2B.state_dict(), 'saved_model/best_netG_A2B.pth')
        torch.save(netG_B2A.state_dict(), 'saved_model/best_netG_B2A.pth')

        # (Optional) Save discriminator and optimizer states for resuming training
        torch.save(netD_A.state_dict(), 'saved_model/best_netD_A.pth')
        torch.save(netD_B.state_dict(), 'saved_model/best_netD_B.pth')
        torch.save(optimizer_G.state_dict(), 'saved_model/best_optimizer_G.pth')
        torch.save(optimizer_D_A.state_dict(), 'saved_model/best_optimizer_D_A.pth')
        torch.save(optimizer_D_B.state_dict(), 'saved_model/best_optimizer_D_B.pth')

Starting training loop...
Epoch [0/20], Batch [100/5000], Loss D: 0.6571, Loss G: 10.8343
Epoch [0/20], Batch [200/5000], Loss D: 0.2860, Loss G: 7.7417
Epoch [0/20], Batch [300/5000], Loss D: 0.4110, Loss G: 9.5730
Epoch [0/20], Batch [400/5000], Loss D: 0.4831, Loss G: 9.1903
Epoch [0/20], Batch [500/5000], Loss D: 0.3283, Loss G: 8.8523
Epoch [0/20], Batch [600/5000], Loss D: 0.4141, Loss G: 5.5280
Epoch [0/20], Batch [700/5000], Loss D: 0.6335, Loss G: 6.1825
Epoch [0/20], Batch [800/5000], Loss D: 0.2360, Loss G: 7.2388
Epoch [0/20], Batch [900/5000], Loss D: 0.2295, Loss G: 7.5491
Epoch [0/20], Batch [1000/5000], Loss D: 0.2994, Loss G: 7.0536
Epoch [0/20], Batch [1100/5000], Loss D: 0.1957, Loss G: 6.2739
Epoch [0/20], Batch [1200/5000], Loss D: 0.3998, Loss G: 5.8267
Epoch [0/20], Batch [1300/5000], Loss D: 0.6803, Loss G: 6.4588
Epoch [0/20], Batch [1400/5000], Loss D: 0.2421, Loss G: 8.1222
Epoch [0/20], Batch [1500/5000], Loss D: 0.6964, Loss G: 6.9432
Epoch [0/20], Batch [1